In [1]:
result=0
for i in range(10):
    result+=100*(1+6.5/100)**i
print(result)

1349.4422542605537


In [2]:
import pandas as pd
import datetime as dt
invest=([100,'2020-5-1'],
        [150,'2021-1-1'],
        [180,'2021-8-1'])
rate_year=0.12
target_time='2022-12-31'
df=pd.DataFrame(invest,columns=['投资额','投资时间'])
df['投资时间']=pd.to_datetime(df['投资时间'])
target=dt.datetime.strptime(target_time,'%Y-%m-%d')
df['时间间隔']=(target-df['投资时间']).dt.days
df

,投资额,投资时间,时间间隔
0,100,2020-05-01,974
1,150,2021-01-01,729
2,180,2021-08-01,517


In [3]:
rate=(1+rate_year)**(1/365)-1
df['系数']=(1+rate)**df['时间间隔']
df['时间价值']=df['投资额']*df['系数']
df

,投资额,投资时间,时间间隔,系数,时间价值
0,100,2020-05-01,974,1.353125,135.312512
1,150,2021-01-01,729,1.254011,188.101587
2,180,2021-08-01,517,1.174125,211.342480


In [4]:
def TVM(rate_year,target_time,*invest):
    rate=(1+rate_year)**(1/365)-1
    target=dt.datetime.strptime(target_time,'%Y-%m-%d')
    df=pd.DataFrame(invest,columns=['投资额','投资时间'])
    df['投资时间']=pd.to_datetime(df['投资时间'])
    df['间隔时间']=(target-df['投资时间']).dt.days
    df['系数']=(1+rate)**df['间隔时间']
    df['时间价值']=df['投资额']*df['系数']
    return df,df['时间价值'].sum()

In [5]:
df1,result1=TVM(0.12,'2022-12-31',[100,'2020-5-1'],[150,'2021-1-1'],[180,'2021-8-1'])
df1
result1

np.float64(534.7565797321254)

In [6]:
import pandas as pd
pd.options.display.float_format='{:,.2f}'.format
data={'投资成本':[1000000,1000000,0,0,0,0,0],
      '销售收入':[0,0,2000000,2000000,2000000,2000000,2000000],
      '付现成本':[0,0,1200000,1230000,1260000,1290000,1320000],
      '折旧':[0,0,400000,400000,400000,400000,400000],}
df=pd.DataFrame(data)
df

,投资成本,销售收入,付现成本,折旧
0,1000000,0,0,0
1,1000000,0,0,0
2,0,2000000,1200000,400000
3,0,2000000,1230000,400000
4,0,2000000,1260000,400000
5,0,2000000,1290000,400000
6,0,2000000,1320000,400000


In [7]:
df['营业利润']=df['销售收入']-df['付现成本']-df['折旧']
df['所得税']=df['营业利润']*0.25
df['税后营业利润']=df['营业利润']-df['所得税']
df['现金净流量']=df['税后营业利润']+df['折旧']-df['投资成本']
df['折现系数']=(1+0.1)**df.index
df['现金流折现']=df['现金净流量']/df['折现系数']
df

,投资成本,销售收入,付现成本,折旧,营业利润,所得税,税后营业利润,现金净流量,折现系数,现金流折现
0,1000000,0,0,0,0,0.00,0.00,"-1,000,000.00",1.00,"-1,000,000.00"
1,1000000,0,0,0,0,0.00,0.00,"-1,000,000.00",1.10,"-909,090.91"
2,0,2000000,1200000,400000,400000,"100,000.00","300,000.00","700,000.00",1.21,"578,512.40"
3,0,2000000,1230000,400000,370000,"92,500.00","277,500.00","677,500.00",1.33,"509,015.78"
4,0,2000000,1260000,400000,340000,"85,000.00","255,000.00","655,000.00",1.46,"447,373.81"
5,0,2000000,1290000,400000,310000,"77,500.00","232,500.00","632,500.00",1.61,"392,732.74"
6,0,2000000,1320000,400000,280000,"70,000.00","210,000.00","610,000.00",1.77,"344,329.10"


In [8]:
NPV=df['现金流折现'].sum()
NPV
df[['现金净流量','现金流折现']].cumsum()

,现金净流量,现金流折现
0,"-1,000,000.00","-1,000,000.00"
1,"-2,000,000.00","-1,909,090.91"
2,"-1,300,000.00","-1,330,578.51"
3,"-622,500.00","-821,562.73"
4,"32,500.00","-374,188.92"
5,"665,000.00","18,543.82"
6,"1,275,000.00","362,872.91"


In [9]:
from sympy import *
y=0
for i, cf in df['现金净流量'].items():
    x=Symbol("x")
    y=cf/(1+x)**i+y
print(y)
y
def newton(y,x0=0.001,e=1e-6):
    x_n=x0-(y.subs(x,x0)/diff(y).subs(x,x0))
    while abs(x_n-x0)>e:
        x0=x_n
        x_n=x0-(y.subs(x,x0)/diff(y).subs(x,x0))
    return x_n
newton(y)

-1000000.0 - 1000000.0/(x + 1) + 700000.0/(x + 1)**2 + 677500.0/(x + 1)**3 + 655000.0/(x + 1)**4 + 632500.0/(x + 1)**5 + 610000.0/(x + 1)**6


0.161068393862569